# Практическая работа: классификация военных и гражданских самолётов

**Дисциплина:** Интеллектуальные системы и технологии  
**Выполнили:** Абрамов Сергей, Ахундов Орхан, Руднев Илья, Марков Никита.  
**Цель:** Сравнить качество бинарной классификации изображений самолётов двумя мультимодальными моделями.

В эксперименте используются **SmolVLM2-500M** и **SmolVLM2-256M**. В отличие от исходного варианта, модели не загружаются в Python через `Transformers`: ноутбук отправляет изображения и текстовые запросы в локальный HTTP API сервера **Ollama**.

**Данные:** 30 фотографий самолётов (15 гражданских и 15 военных), включая нетривиальные варианты. Изображения находятся в `dataset/`, эталонные классы и категории — в `metadata.csv`. Исходный набор передаётся отдельным ZIP-архивом `aircraft_dataset_for_classmate.zip`.

**Метрики:** Accuracy, Precision, Recall, F1 (положительный класс — `military`), точность на сложных примерах, средняя задержка ответа, матрицы ошибок.

> Важно: через Ollama используются GGUF-сборки моделей с квантованием Q8_0. Поэтому результаты нового запуска нужно измерить заново — они не обязаны совпасть с результатами исходного запуска моделей в Transformers на GPU.

## 1. Установка зависимостей и загрузка датасета

Выполните ячейку установки библиотек. Затем в следующей ячейке выберите ZIP-архив с датасетом через стандартный виджет Jupyter. Если виджет загрузки в используемой среде не поддерживается, укажите путь к архиву вручную в ячейке распаковки.

In [1]:
%pip install -q requests ipywidgets scikit-learn matplotlib seaborn pillow tqdm pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

METADATA_PATH = Path("metadata.csv")
DATASET_DIR = Path("dataset")


# Загрузка и проверка датасета

if not METADATA_PATH.is_file():
    raise FileNotFoundError(f"Не найден metadata.csv: {METADATA_PATH}")

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Не найдена папка dataset: {DATASET_DIR}")

df = pd.read_csv(METADATA_PATH)

# Проверяем необходимые столбцы
required_columns = {"filename", "true_label"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"В metadata.csv отсутствуют столбцы: {missing_columns}")

# Если отсутствует название самолёта — используем имя файла
if "aircraft_name" not in df.columns:
    df["aircraft_name"] = df["filename"]

# Проверяем изображения
def find_image(filename):
    filename = str(filename)

    # Проверка пути относительно dataset
    path = DATASET_DIR / filename

    if path.is_file():
        return str(path.resolve())

    # Поиск по вложенным папкам
    matches = list(DATASET_DIR.rglob(Path(filename).name))

    if len(matches) == 1:
        return str(matches[0].resolve())

    return None

df["_image_path"] = df["filename"].apply(find_image)

missing_images = df[df["_image_path"].isna()]["filename"].tolist()

if missing_images:
    raise FileNotFoundError(
        f"Не найдены изображения: {missing_images}"
    )

print(f"Датасет загружен: {len(df)} изображений")
print(f"Гражданские: {(df['true_label'] == 'civilian').sum()}")
print(f"Военные: {(df['true_label'] == 'military').sum()}")

display(df.head())

Датасет загружен: 30 изображений
Гражданские: 15
Военные: 15


,filename,true_label,category,aircraft_name,id,_image_path
0,C01.jpg,civilian,basic_civilian,Airbus A320 family,C01,D:\Институт\Магистратура\Интеллектуальные сист...
1,C02.jpg,civilian,basic_civilian,Boeing 737,C02,D:\Институт\Магистратура\Интеллектуальные сист...
2,C03.jpg,civilian,basic_civilian,Airbus A380,C03,D:\Институт\Магистратура\Интеллектуальные сист...
3,C04.jpg,civilian,basic_civilian,Boeing 747,C04,D:\Институт\Магистратура\Интеллектуальные сист...
4,C05.jpg,civilian,basic_civilian,Boeing 787 Dreamliner,C05,D:\Институт\Магистратура\Интеллектуальные сист...


## 2. Подключение к Ollama API

Ноутбук обращается к REST API Ollama по адресу `http://localhost:11434`. Перед запуском эксперимента:

1. Установите Ollama с официального сайта: <https://ollama.com/download>.
2. Запустите Ollama. Обычно сервер API автоматически становится доступен на порту `11434`.
3. В терминале выполните команды для загрузки двух моделей:

```bash
ollama pull hf.co/ggml-org/SmolVLM2-500M-Video-Instruct-GGUF:Q8_0
ollama pull hf.co/ggml-org/SmolVLM2-256M-Video-Instruct-GGUF:Q8_0
```

4. Проверьте, что модели появились в списке командой `ollama list`.

В ноутбуке не требуется GPU, `torch`, `transformers` или токен Hugging Face. Изображения передаются в API в формате Base64, а ответ модели возвращается в JSON. Такой формат передачи изображений соответствует документации Ollama: <https://docs.ollama.com/capabilities/vision>.

**Важно:** если Jupyter работает не на том же компьютере, что и Ollama, `localhost` будет указывать на сервер Jupyter. В таком случае укажите доступный сетевой адрес Ollama в переменной `OLLAMA_BASE_URL` и настройте сетевой доступ отдельно.

In [9]:
import base64
import re
import time
import requests
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

# Адрес REST API Ollama
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_CHAT_URL = f"{OLLAMA_BASE_URL.rstrip('/')}/api/chat"

# Используются те же семейства моделей, что и в исходном эксперименте
# Указаны GGUF-сборки Q8_0, доступные для запуска через Ollama
MODELS = {
    "SmolVLM2-500M":
        "hf.co/ggml-org/SmolVLM2-500M-Video-Instruct-GGUF:Q8_0",
    "SmolVLM2-256M":
        "hf.co/ggml-org/SmolVLM2-256M-Video-Instruct-GGUF:Q8_0"
}

PROMPT = (
    "Look at the aircraft in this image. "
    "Is it military or civilian? "
    "Answer with exactly one lowercase word: "
    "military or civilian."
)

# Проверяем доступность сервера и выводим список загруженных моделей
try:
    health_response = requests.get(
        f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags",
        timeout=10
    )
    health_response.raise_for_status()
    server_info = health_response.json()

    print("Подключение к Ollama API успешно.")
    installed_models = [
        model.get("name", "")
        for model in server_info.get("models", [])
    ]
    print("Модели, доступные в Ollama:")
    for model in installed_models:
        print(" -", model)

except requests.exceptions.RequestException as exc:
    raise RuntimeError(
        "Не удалось подключиться к Ollama API. "
        "Убедитесь, что Ollama установлена и запущена, "
        f"а адрес OLLAMA_BASE_URL указан верно. Подробности: {exc}"
    ) from exc

Подключение к Ollama API успешно.
Модели, доступные в Ollama:
 - hf.co/ggml-org/SmolVLM2-256M-Video-Instruct-GGUF:Q8_0
 - hf.co/ggml-org/SmolVLM2-500M-Video-Instruct-GGUF:Q8_0


## 3. Методика эксперимента

Каждому изображению передаётся одинаковая англоязычная инструкция: определить один из двух классов (`civilian` / `military`). Каждая модель обрабатывает все изображения в одном и том же порядке.

Для воспроизводимости используется `temperature=0`. Ответ сохраняется после обработки каждого снимка в `evaluation_results.csv`. Время измеряется на стороне ноутбука и включает полный HTTP-запрос к локальному серверу Ollama.

При технической ошибке записывается `error`, при ответе, из которого нельзя однозначно извлечь класс, — `unknown`. Такие записи не считаются корректными предсказаниями. Перед сравнением метрик необходимо проверить, что обе модели дали по 30 допустимых ответов.

In [ ]:
def classify_with_ollama(image_path: Path, model_id: str) -> str:
    # Отправляет одно изображение в Ollama API и возвращает текст ответа модели.
    image_path = Path(row["_image_path"])

    if not image_path.is_file():
        raise FileNotFoundError(f"Изображение не найдено: {image_path}")

    image_base64 = base64.b64encode(
        image_path.read_bytes()
    ).decode("utf-8")

    payload = {
        "model": model_id,
        "messages": [
            {
                "role": "user",
                "content": PROMPT,
                "images": [image_base64]
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 12
        }
    }

    response = requests.post(
        OLLAMA_CHAT_URL,
        json=payload,
        timeout=(15, 300)
    )
    response.raise_for_status()

    response_data = response.json()
    return response_data["message"]["content"].strip()

def parse_label(raw_answer: str) -> str:
    # Извлекает civilian/military, если ответ содержит один однозначный класс.
    found = re.findall(
        r"\b(?:military|civilian)\b",
        str(raw_answer).lower()
    )

    if found and len(set(found)) == 1:
        return found[0]

    return "unknown"


# Основной эксперимент: по умолчанию обрабатываются все строки датасета.
MAX_IMAGES = 30

if len(df) < MAX_IMAGES:
    print(
        f"Внимание: в датасете только {len(df)} изображений, "
        f"поэтому будут обработаны все доступные."
    )

results_df = df.head(MAX_IMAGES).copy()

# Создаём колонки заранее, чтобы типы данных были стабильными.
for model_name in MODELS:
    results_df[f"{model_name}_pred"] = ""
    results_df[f"{model_name}_reason"] = ""
    results_df[f"{model_name}_time"] = 0.0

for model_name, model_id in MODELS.items():
    print(f"\nЗапуск модели: {model_name}")

    for idx, row in tqdm(
        results_df.iterrows(),
        total=len(results_df),
        desc=model_name
    ):
        image_path = DATASET_DIR / str(row["filename"])
        start_time = time.perf_counter()

        try:
            raw_answer = classify_with_ollama(image_path, model_id)
            predicted_label = parse_label(raw_answer)

        except Exception as exc:
            predicted_label = "error"
            raw_answer = f"{type(exc).__name__}: {exc}"

            print("\nОШИБКА API:")
            print(raw_answer)

        elapsed_time = round(time.perf_counter() - start_time, 2)

        results_df.loc[idx, f"{model_name}_pred"] = predicted_label
        results_df.loc[idx, f"{model_name}_reason"] = raw_answer
        results_df.loc[idx, f"{model_name}_time"] = elapsed_time

        # Промежуточное сохранение защищает результаты от потери при прерывании.
        results_df.to_csv("evaluation_results.csv", index=False)

        print(
            f"{row['filename']} | "
            f"Истина: {row['true_label']} | "
            f"Ответ: {predicted_label} | "
            f"Время: {elapsed_time} с"
        )

display(results_df.head())

## 4. Проверка полноты результатов

Проверяется, что каждая модель классифицировала все изображения без `error`, `unknown` и пропусков. Если количество допустимых предсказаний меньше размера выборки, метрики будут рассчитаны только по успешно классифицированным изображениям, поэтому сравнение следует интерпретировать с учётом технических ошибок.

In [ ]:
for model_name in MODELS:
    column = f"{model_name}_pred"

    print(f"\nМодель: {model_name}")
    print(results_df[column].value_counts(dropna=False))

    valid_count = results_df[column].isin(
        ["civilian", "military"]
    ).sum()

    print(
        f"Успешных классификаций: "
        f"{valid_count}/{len(results_df)}"
    )

    if valid_count != len(results_df):
        print(
            "Внимание: есть технические ошибки или неоднозначные ответы. "
            "Проверьте столбец с ответами модели перед интерпретацией метрик."
        )

## 5. Количественное сравнение

**Accuracy** — доля верно классифицированных изображений. **Precision** для класса `military` — доля действительно военных среди предсказанных военных; **Recall** — доля обнаруженных военных среди всех военных; **F1** — гармоническое среднее Precision и Recall.

«Сложные случаи» — записи, категория которых отличается от `basic_civilian` и `basic_military`. Технические ошибки и неоднозначные ответы исключаются из расчёта метрик классификации и отдельно учитываются при проверке полноты.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def calculate_metrics(data: pd.DataFrame, model_prefix: str) -> dict:
    pred_column = f"{model_prefix}_pred"
    time_column = f"{model_prefix}_time"

    valid_data = data[
        data[pred_column].isin(["military", "civilian"])
    ].copy()

    if valid_data.empty:
        return {
            "Model": model_prefix,
            "Accuracy (All)": np.nan,
            "Accuracy (Hard Cases)": np.nan,
            "Precision (Military)": np.nan,
            "Recall (Military)": np.nan,
            "F1-Score": np.nan,
            "Avg Latency (s)": np.nan
        }

    y_true = valid_data["true_label"]
    y_pred = valid_data[pred_column]

    hard_data = valid_data[
        ~valid_data["category"].isin(
            ["basic_civilian", "basic_military"]
        )
    ]

    hard_accuracy = (
        accuracy_score(hard_data["true_label"], hard_data[pred_column])
        if not hard_data.empty
        else np.nan
    )

    return {
        "Model": model_prefix,
        "Accuracy (All)": round(accuracy_score(y_true, y_pred), 3),
        "Accuracy (Hard Cases)": (
            round(hard_accuracy, 3)
            if not np.isnan(hard_accuracy)
            else np.nan
        ),
        "Precision (Military)": round(
            precision_score(
                y_true, y_pred,
                pos_label="military",
                zero_division=0
            ),
            3
        ),
        "Recall (Military)": round(
            recall_score(
                y_true, y_pred,
                pos_label="military",
                zero_division=0
            ),
            3
        ),
        "F1-Score": round(
            f1_score(
                y_true, y_pred,
                pos_label="military",
                zero_division=0
            ),
            3
        ),
        "Avg Latency (s)": round(
            valid_data[time_column].astype(float).mean(),
            2
        )
    }


metrics_list = [
    calculate_metrics(results_df, model_name)
    for model_name in MODELS
]

metrics_table = pd.DataFrame(metrics_list).set_index("Model")
display(metrics_table)

## 6. Матрицы ошибок

По строкам указан истинный класс, по столбцам — предсказанный. Числа на диагонали отражают правильные ответы, вне диагонали — ошибки. В матрицы включаются только допустимые предсказания `civilian` и `military`.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 4.5))

if len(MODELS) == 1:
    axes = [axes]

labels = ["civilian", "military"]

for ax, model_name in zip(axes, MODELS):
    pred_column = f"{model_name}_pred"

    valid_data = results_df[
        results_df[pred_column].isin(labels)
    ]

    if valid_data.empty:
        ax.set_title(f"{model_name}: нет допустимых ответов")
        ax.axis("off")
        continue

    cm = confusion_matrix(
        valid_data["true_label"],
        valid_data[pred_column],
        labels=labels
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
        cbar=False
    )

    ax.set_title(f"Confusion Matrix: {model_name}")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

plt.tight_layout()
plt.show()

## 7. Анализ ошибочных классификаций

Анализируются самолёты, для которых допустимый ответ модели (`civilian` или `military`) отличается от эталонной метки. Технические ошибки и неоднозначные ответы `unknown` не включаются в список ошибок классификации.

In [ ]:
for model_name in MODELS:
    print(f"\nОшибки модели {model_name}")

    pred_column = f"{model_name}_pred"

    mistakes = results_df[
        results_df[pred_column].isin(["civilian", "military"])
        & (results_df["true_label"] != results_df[pred_column])
    ]

    if mistakes.empty:
        print("Ошибок классификации нет.")
    else:
        display(mistakes[[
            "aircraft_name",
            "category",
            "true_label",
            pred_column
        ]])

    invalid = results_df[
        ~results_df[pred_column].isin(["civilian", "military"])
    ]

    if not invalid.empty:
        print("Технические ошибки или неоднозначные ответы:")
        display(invalid[[
            "aircraft_name",
            "filename",
            pred_column,
            f"{model_name}_reason"
        ]])

## 8. Выводы по результатам запуска

Результаты формируются автоматически на основании текущего запуска. Это позволяет не использовать в выводах старые значения, полученные при локальном запуске моделей через Transformers в Google Colab.

In [ ]:
print(f"Размер выборки: {len(results_df)} изображений")
print("\nСводные метрики:")
display(metrics_table)

if metrics_table["Accuracy (All)"].notna().any():
    best_model = metrics_table["Accuracy (All)"].idxmax()
    best_accuracy = metrics_table.loc[best_model, "Accuracy (All)"]

    print(
        f"Наибольшая Accuracy в этом запуске у модели "
        f"{best_model}: {best_accuracy:.3f}."
    )
else:
    print(
        "Не удалось рассчитать Accuracy: модели не вернули "
        "допустимых ответов. Проверьте подключение к Ollama и журнал ошибок."
    )

print(
    "\nПримечание: модели вызываются через локальный API Ollama "
    "и используются в GGUF-квантовании Q8_0. "
    "Задержка включает время HTTP-запроса."
)
print(
    "Результаты небольшой выборки из 30 изображений не следует "
    "обобщать на все типы гражданской и военной авиации."
)

## 9. Сохранение результатов

Сохраняются два CSV-файла:

- `evaluation_results.csv` — исходная разметка, ответы обеих моделей, текст ответов и время обработки;
- `model_metrics.csv` — итоговые метрики качества.

Для воспроизведения эксперимента необходимо передать ноутбук, ZIP-архив с датасетом и обеспечить доступ к Ollama с загруженными моделями.

**Полезные ссылки:**
- [Документация Ollama API](https://docs.ollama.com/api)
- [Передача изображений в Ollama](https://docs.ollama.com/capabilities/vision)
- [SmolVLM2-500M GGUF](https://huggingface.co/ggml-org/SmolVLM2-500M-Video-Instruct-GGUF)
- [SmolVLM2-256M GGUF](https://huggingface.co/ggml-org/SmolVLM2-256M-Video-Instruct-GGUF)

In [ ]:
results_df.to_csv("evaluation_results.csv", index=False)
metrics_table.to_csv("model_metrics.csv")

print("Созданы файлы:")
print(" - evaluation_results.csv")
print(" - model_metrics.csv")